<a href="https://colab.research.google.com/github/treborskrub/Modular-/blob/main/LinguisticMapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

@dataclass
class EvidenceQuantum:
    """Atomic unit of grounding evidence."""
    claim_fragment: str
    support_score: float          # 0.0 to 1.0
    confidence: float             # source reliability

class LinguisticMapper:
    def __init__(self, phi_base: float = (1 + 5**0.5) / 2) -> None:
        self.phi_base = phi_base

    def extract_claims(self, text: str) -> List[str]:
        """Splits full text into individual auditable claims."""
        sentences = re.split(r'(?<=[.!?])\s+', text.strip())
        return [s.strip() for s in sentences if len(s.strip()) > 10]

    def measure_assertion_confidence(self, claim: str) -> float:
        """Determines assertion strength via marker words."""
        words = claim.lower().split()
        strong_markers = {"is", "are", "always", "never", "definitely", "certainly", "proven", "fact"}
        weak_markers = {"might", "could", "perhaps", "maybe", "typically", "usually"}

        strong_count = sum(1 for w in words if w in strong_markers)
        weak_count = sum(1 for w in words if w in weak_markers)

        base = 0.5 + (strong_count * 0.08) - (weak_count * 0.06)
        return min(1.0, max(0.1, base))

    def accumulate_grounding(self, evidence: Optional[List[EvidenceQuantum]]) -> float:
        """Scales evidence using geometric decay with golden ratio."""
        if not evidence:
            return 0.0

        total_support = 0.0
        for idx, eq in enumerate(evidence):
            weight = self.phi_base ** (-(idx + 1))
            total_support += eq.support_score * eq.confidence * (1 - weight)

        return min(1.0, total_support / self.phi_base)